In [ ]:
import csv
import re
import time

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
    NoSuchElementException,
    StaleElementReferenceException,
    NoSuchWindowException
)

from webdriver_manager.chrome import ChromeDriverManager

# INSTAGRAM LOGIN
USERNAME = "Enter_your_username_here"
PASSWORD = "Enter_your_password_here"


# Total scraping duration
SCRAPE_DURATION_MINUTES = 30

BATCH_SIZE = 10


PAGE_LOAD_TIMEOUT = 30
ELEMENT_WAIT_TIMEOUT = 20

REQUEST_DELAY = 2

SCROLL_DELAY = 2

CAPTCHA_WAIT_SECONDS = 30


timestamp_str = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

OUTPUT_FILE = (
    f"{timestamp_str}_instagram_research_data.csv"
)


CSV_FIELDS = [
    "type",
    "url",
    "caption",
    "hashtags",
    "mentions",
    "timestamp",
    "location"
]


def create_driver():

    options = webdriver.ChromeOptions()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")

    driver = webdriver.Chrome(
        service=Service(
            ChromeDriverManager().install()
        ),
        options=options
    )

    driver.set_page_load_timeout(
        PAGE_LOAD_TIMEOUT
    )

    return driver



def create_csv_file():

    with open(
        OUTPUT_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=CSV_FIELDS
        )

        writer.writeheader()


def append_row_to_csv(row):

    try:

        with open(
            OUTPUT_FILE,
            "a",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.DictWriter(
                file,
                fieldnames=CSV_FIELDS
            )

            writer.writerow(row)

    except Exception as e:

        print(
            f"CSV save error: {e}"
        )


# ============================================================
# CHECK BROWSER
# ============================================================

def browser_is_alive(driver):

    try:

        driver.current_url

        driver.window_handles

        return True

    except (
        NoSuchWindowException,
        WebDriverException
    ):

        return False

    except Exception:

        return False


def login(driver):

    try:

        print("\nOpening Instagram login page...")

        driver.get(
            "https://www.instagram.com/accounts/login/"
        )

        wait = WebDriverWait(
            driver,
            ELEMENT_WAIT_TIMEOUT
        )


       
        username_field = wait.until(
            EC.presence_of_element_located(
                (
                    By.CSS_SELECTOR,
                    "input[name='email']"
                )
            )
        )

        username_field.clear()

        username_field.send_keys(
            USERNAME
        )

        print(
            "Username entered."
        )

        password_field = wait.until(
            EC.presence_of_element_located(
                (
                    By.CSS_SELECTOR,
                    "input[name='pass']"
                )
            )
        )

        password_field.clear()

        password_field.send_keys(
            PASSWORD
        )

        print(
            "Password entered."
        )

        login_button = wait.until(
            EC.presence_of_element_located(
                (
                    By.CSS_SELECTOR,
                    "input[type='submit']"
                )
            )
        )

        driver.execute_script(
            """
            arguments[0].scrollIntoView({
                block: 'center'
            });
            """,
            login_button
        )

        time.sleep(1)

        driver.execute_script(
            "arguments[0].click();",
            login_button
        )

        print(
            "Login button clicked."
        )

        print("\n")
        print("=" * 65)

        print(
            "MANUAL CAPTCHA / SECURITY VERIFICATION"
        )

        print("=" * 65)

        print(
            "If Instagram displays a CAPTCHA, "
            "security check, or verification,"
        )

        print(
            "complete it manually in the Chrome window."
        )

        print(
            f"You have {CAPTCHA_WAIT_SECONDS} seconds."
        )

        print(
            "The scraper will NOT attempt to bypass it."
        )

        print("=" * 65)


        for remaining in range(
            CAPTCHA_WAIT_SECONDS,
            0,
            -1
        ):

            print(
                f"\rTime remaining: "
                f"{remaining:02d} seconds",
                end="",
                flush=True
            )

            time.sleep(1)

        print("\n")

        print(
            "CAPTCHA/verification window finished."
        )

        print(
            "Checking Instagram login session..."
        )

        time.sleep(3)


        # ====================================================
        # CHECK CURRENT URL
        # ====================================================

        current_url = driver.current_url.lower()

        print(
            f"Current URL: {driver.current_url}"
        )

        if "accounts/login" in current_url:

            print("\nLOGIN NOT CONFIRMED.")

            print(
                "Instagram is still showing "
                "the login page."
            )

            print(
                "Explore will NOT be opened."
            )

            return False

        try:

            login_fields = driver.find_elements(
                By.CSS_SELECTOR,
                "input[name='email'], input[name='pass']"
            )

        except Exception:

            login_fields = []


        if login_fields:

            print("\nLOGIN NOT CONFIRMED.")

            print(
                "Instagram login fields are still present."
            )

            print(
                "Explore will NOT be opened."
            )

            return False

        print("\n")
        print("=" * 65)

        print(
            "LOGIN CONFIRMED"
        )

        print("=" * 65)

        print(
            "Authenticated Instagram session detected."
        )

        return True


    except TimeoutException as e:

        print("\nLOGIN TIMEOUT.")

        print(e)

        return False


    except Exception as e:

        print(
            f"\nLOGIN FAILED: {e}"
        )

        return False

def open_explore(driver):

    try:

        print("\n")

        print(
            "Opening personalized Instagram Explore..."
        )

        driver.get(
            "https://www.instagram.com/explore/"
        )

        wait = WebDriverWait(
            driver,
            ELEMENT_WAIT_TIMEOUT
        )

        wait.until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "body")
            )
        )

        time.sleep(4)


        current_url = driver.current_url.lower()

        if "accounts/login" in current_url:

            print(
                "Instagram redirected to login."
            )

            print(
                "Authenticated session was not maintained."
            )

            return False


        print(
            "Personalized Explore page loaded."
        )

        return True


    except Exception as e:

        print(
            f"Could not open Explore: {e}"
        )

        return False


def find_new_links(
    driver,
    seen_links,
    required_count
):

    new_links = []


    try:

        anchors = driver.find_elements(
            By.TAG_NAME,
            "a"
        )


        for anchor in anchors:

            try:

                href = anchor.get_attribute(
                    "href"
                )

                if not href:
                    continue


                # ------------------------------------------------
                # POST
                # ------------------------------------------------

                if "/p/" in href:

                    clean_url = (
                        href.split("?")[0]
                    )


                # ------------------------------------------------
                # REEL
                # ------------------------------------------------

                elif "/reel/" in href:

                    clean_url = (
                        href.split("?")[0]
                    )


                else:

                    continue


                # ------------------------------------------------
                # DUPLICATE CHECK
                # ------------------------------------------------

                if clean_url in seen_links:
                    continue

                if clean_url in new_links:
                    continue

                new_links.append(
                    clean_url
                )

                seen_links.add(
                    clean_url
                )


                print(
                    f"Found "
                    f"{len(new_links)}/{required_count}: "
                    f"{clean_url}"
                )


                if len(new_links) >= required_count:

                    break


            except (
                StaleElementReferenceException,
                WebDriverException
            ):

                continue


            except Exception:

                continue


    except Exception as e:

        print(
            f"Error finding links: {e}"
        )


    return new_links

def collect_next_batch(
    driver,
    seen_links,
    batch_size,
    end_time
):

    batch_links = []

    no_new_link_attempts = 0


    while (
        len(batch_links) < batch_size
        and time.time() < end_time
    ):

        if not browser_is_alive(driver):

            print(
                "\nChrome/Instagram window was closed."
            )

            return batch_links


        remaining_required = (
            batch_size - len(batch_links)
        )


        new_links = find_new_links(
            driver,
            seen_links,
            remaining_required
        )


        if new_links:

            batch_links.extend(
                new_links
            )

            no_new_link_attempts = 0

        else:

            no_new_link_attempts += 1


        if len(batch_links) >= batch_size:

            break


        if time.time() >= end_time:

            break

        try:

            driver.execute_script(
                """
                window.scrollBy({
                    top: 1200,
                    behavior: 'smooth'
                });
                """
            )

        except (
            NoSuchWindowException,
            WebDriverException
        ):

            print(
                "\nChrome/Instagram window was closed."
            )

            return batch_links

        except Exception as e:

            print(
                f"Scroll error: {e}"
            )


        time.sleep(
            SCROLL_DELAY
        )


        if no_new_link_attempts >= 10:

            print(
                "\nNo new links found after "
                "multiple scroll attempts."
            )

            break


    return batch_links


def get_meta_content(
    driver,
    selector
):

    try:

        element = driver.find_element(
            By.CSS_SELECTOR,
            selector
        )

        return (
            element.get_attribute(
                "content"
            ) or ""
        )


    except (
        NoSuchElementException,
        StaleElementReferenceException
    ):

        return ""


    except Exception:

        return ""


def clean_caption_from_meta(text):

    if not text:

        return ""

    text = text.strip()


    match = re.search(
        r':\s*"(.+)"$',
        text,
        re.DOTALL
    )


    if match:

        return match.group(1).strip()


    return text

def extract_caption(driver):

    candidates = []

    description = get_meta_content(
        driver,
        "meta[property='og:description']"
    )


    if description:

        cleaned = (
            clean_caption_from_meta(
                description
            )
        )

        if cleaned:

            candidates.append(
                cleaned
            )

    title = get_meta_content(
        driver,
        "meta[property='og:title']"
    )


    if title:

        cleaned = (
            clean_caption_from_meta(
                title
            )
        )

        if cleaned:

            candidates.append(
                cleaned
            )


    selectors = [
        "article h1",
        "article div[dir='auto']"
    ]


    for selector in selectors:

        try:

            elements = driver.find_elements(
                By.CSS_SELECTOR,
                selector
            )


            for element in elements:

                try:

                    text = element.text.strip()


                    if (
                        text
                        and len(text) > 5
                    ):

                        candidates.append(
                            text
                        )


                except Exception:

                    continue


        except Exception:

            continue

    for candidate in candidates:

        if candidate:

            return candidate


    return ""


def extract_timestamp(driver):

    try:

        time_elements = driver.find_elements(
            By.TAG_NAME,
            "time"
        )


        for element in time_elements:

            try:

                timestamp = (
                    element.get_attribute(
                        "datetime"
                    )
                )


                if timestamp:

                    return timestamp


            except Exception:

                continue


    except Exception:

        pass


    return ""

def extract_location(driver):

    try:

        links = driver.find_elements(
            By.TAG_NAME,
            "a"
        )


        for link in links:

            try:

                href = link.get_attribute(
                    "href"
                )

                text = link.text.strip()


                if (
                    href
                    and "/explore/locations/" in href
                    and text
                ):

                    return text


            except Exception:

                continue


    except Exception:

        pass


    return ""


def scrape_item_same_window(
    driver,
    url,
    explore_scroll_position
):

    try:

        explore_url = (
            "https://www.instagram.com/explore/"
        )


        print(
            "Opening item in the same Chrome tab..."
        )


        driver.get(url)

        wait = WebDriverWait(
            driver,
            ELEMENT_WAIT_TIMEOUT
        )


        wait.until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "body")
            )
        )


        time.sleep(2)

        caption = extract_caption(
            driver
        )

        hashtags = re.findall(
            r"#\w+",
            caption
        )

        mentions = re.findall(
            r"@\w+",
            caption
        )

        timestamp = extract_timestamp(
            driver
        )

        location = extract_location(
            driver
        )


        data = {

            "caption": caption,

            "hashtags": ", ".join(
                hashtags
            ),

            "mentions": ", ".join(
                mentions
            ),

            "timestamp": timestamp,

            "location": location
        }

        print(
            "Returning to Explore..."
        )


        driver.get(
            explore_url
        )


        wait.until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "body")
            )
        )


        time.sleep(3)

        driver.execute_script(
            """
            window.scrollTo(
                0,
                arguments[0]
            );
            """,
            explore_scroll_position
        )


        time.sleep(1)


        return data


    except Exception as e:

        try:

            driver.get(
                "https://www.instagram.com/explore/"
            )


            WebDriverWait(
                driver,
                ELEMENT_WAIT_TIMEOUT
            ).until(
                EC.presence_of_element_located(
                    (By.TAG_NAME, "body")
                )
            )


            time.sleep(2)


        except Exception:

            pass


        raise e

def scrape_batch(
    driver,
    batch_links,
    batch_number,
    total_saved,
    end_time
):

    successful = 0
    failed = 0


    print("\n")
    print("=" * 65)

    print(
        f"BATCH {batch_number}"
    )

    print("=" * 65)

    print(
        f"Items in batch: "
        f"{len(batch_links)}"
    )

    for index, url in enumerate(
        batch_links,
        start=1
    ):


        if not browser_is_alive(driver):

            print(
                "\nChrome/Instagram window was closed."
            )

            print(
                "Stopping scraper safely."
            )

            return (
                total_saved,
                True
            )


        if time.time() >= end_time:

            print(
                "\nTime limit reached."
            )

            break

        try:

            explore_scroll_position = (
                driver.execute_script(
                    "return window.pageYOffset;"
                )
            )

        except Exception:

            explore_scroll_position = 0


        print("\n")
        print(
            f"Scraping "
            f"{index}/{len(batch_links)}"
        )

        print(url)


        try:

            if "/reel/" in url:

                item_type = "reel"

            else:

                item_type = "post"

            data = scrape_item_same_window(
                driver=driver,
                url=url,
                explore_scroll_position=(
                    explore_scroll_position
                )
            )

            row = {

                "type": item_type,

                "url": url,

                "caption": data[
                    "caption"
                ],

                "hashtags": data[
                    "hashtags"
                ],

                "mentions": data[
                    "mentions"
                ],

                "timestamp": data[
                    "timestamp"
                ],

                "location": data[
                    "location"
                ]
            }

            append_row_to_csv(
                row
            )


            successful += 1

            total_saved += 1


            print(
                "✓ Saved successfully"
            )


        except (
            NoSuchWindowException,
            WebDriverException
        ):

            print(
                "\nChrome/Instagram window was closed."
            )

            print(
                "Stopping scraper safely."
            )

            return (
                total_saved,
                True
            )


        except Exception as e:

            failed += 1


            print(
                "✗ Item failed."
            )

            print(
                f"Error: {e}"
            )

            try:

                if browser_is_alive(driver):

                    driver.get(
                        "https://www.instagram.com/explore/"
                    )

                    WebDriverWait(
                        driver,
                        ELEMENT_WAIT_TIMEOUT
                    ).until(
                        EC.presence_of_element_located(
                            (By.TAG_NAME, "body")
                        )
                    )

                    time.sleep(2)


            except Exception:

                pass

        if time.time() < end_time:

            time.sleep(
                REQUEST_DELAY
            )

    print("\n")
    print("-" * 65)

    print(
        f"BATCH {batch_number} COMPLETE"
    )

    print("-" * 65)

    print(
        f"Successful: {successful}"
    )

    print(
        f"Failed: {failed}"
    )

    print(
        f"Total saved: {total_saved}"
    )

    print("-" * 65)


    return (
        total_saved,
        False
    )

def main():

    driver = None

    total_saved = 0

    batch_number = 0

    start_time = None


    try:

        create_csv_file()


        print("\n")
        print(
            "Data file:"
        )

        print(
            OUTPUT_FILE
        )

        print("\nStarting Chrome...")


        driver = create_driver()

        if not login(driver):

            print("\n")
            print(
                "Could not confirm Instagram login."
            )

            print(
                "Scraper stopped."
            )

            return

        if not open_explore(driver):

            print("\n")
            print(
                "Could not open personalized Explore."
            )

            print(
                "Scraper stopped."
            )

            return

        main_window = (
            driver.current_window_handle
        )


        print("\n")
        print(
            "Single Chrome window established."
        )

        print(
            "No additional browser windows or tabs "
            "will be created."
        )


        start_time = time.time()


        end_time = (
            start_time
            + SCRAPE_DURATION_MINUTES * 60
        )

        seen_links = set()

        print("\n")
        print("=" * 65)

        print(
            "TIME-BASED SCRAPING STARTED"
        )

        print(
            f"Duration: "
            f"{SCRAPE_DURATION_MINUTES} minutes"
        )

        print(
            f"Batch size: "
            f"{BATCH_SIZE}"
        )

        print(
            "Source: Personalized Instagram Explore"
        )

        print(
            "Browser mode: SINGLE WINDOW / SINGLE TAB"
        )

        print("=" * 65)

        while time.time() < end_time:


            if not browser_is_alive(driver):

                print("\n")
                print(
                    "Instagram Chrome window was closed."
                )

                print(
                    "Stopping scraper safely."
                )

                break

            remaining_seconds = (
                end_time - time.time()
            )


            if remaining_seconds <= 0:

                break


            print("\n")
            print("=" * 65)

            print(
                f"Time remaining: "
                f"{remaining_seconds / 60:.2f} minutes"
            )

            print("=" * 65)

            batch_number += 1


            print(
                f"\nCollecting batch "
                f"{batch_number}..."
            )


            batch_links = collect_next_batch(
                driver=driver,
                seen_links=seen_links,
                batch_size=BATCH_SIZE,
                end_time=end_time
            )

            if not browser_is_alive(driver):

                print("\n")
                print(
                    "Instagram Chrome window was closed."
                )

                break

            if not batch_links:

                print(
                    "\nNo new post/reel links found."
                )


                if time.time() >= end_time:

                    break


                try:

                    driver.execute_script(
                        """
                        window.scrollBy({
                            top: 1500,
                            behavior: 'smooth'
                        });
                        """
                    )

                except Exception:

                    break


                time.sleep(
                    SCROLL_DELAY
                )

                continue

            if time.time() >= end_time:

                print(
                    "\nTime limit reached."
                )

                break


            (
                total_saved,
                browser_closed
            ) = scrape_batch(
                driver=driver,
                batch_links=batch_links,
                batch_number=batch_number,
                total_saved=total_saved,
                end_time=end_time
            )


         
            if browser_closed:

                break

        if start_time:

            elapsed_seconds = (
                time.time() - start_time
            )

        else:

            elapsed_seconds = 0


        print("\n")
        print("=" * 65)

        print(
            "SCRAPING FINISHED"
        )

        print("=" * 65)


        print(
            f"Total batches attempted: "
            f"{batch_number}"
        )


        print(
            f"Unique URLs discovered: "
            f"{len(seen_links)}"
        )


        print(
            f"Successful records: "
            f"{total_saved}"
        )


        print(
            f"Runtime: "
            f"{elapsed_seconds / 60:.2f} minutes"
        )


        print("\n")
        print(
            "ALL DATA SAVED TO:"
        )


        print(
            OUTPUT_FILE
        )


        print("\n")
        print(
            "ONE CSV FILE WAS USED FOR THIS RUN."
        )

    except KeyboardInterrupt:

        print("\n")
        print(
            "Scraper stopped manually."
        )

        print(
            f"Records already saved: "
            f"{total_saved}"
        )

        print(
            "\nSaved data remains in:"
        )

        print(
            OUTPUT_FILE
        )

    except (
        NoSuchWindowException,
        WebDriverException
    ):

        print("\n")
        print(
            "Chrome/Instagram window was closed."
        )

        print(
            "Scraper stopped safely."
        )

        print(
            f"Records already saved: "
            f"{total_saved}"
        )

        print(
            "\nSaved data remains in:"
        )

        print(
            OUTPUT_FILE
        )

    except Exception as e:

        print("\n")
        print(
            "Unexpected program error:"
        )

        print(
            e
        )

        print(
            f"\nRecords already saved: "
            f"{total_saved}"
        )

        print(
            "\nSaved data remains in:"
        )

        print(
            OUTPUT_FILE
        )

    finally:

        if driver:

            try:

                driver.quit()

            except Exception:

                pass


            print(
                "Browser session closed."
            )

if __name__ == "__main__":

    main()



Data file:
20260811_204611_instagram_research_data.csv

Starting Chrome...

Opening Instagram login page...
Username entered.
Password entered.
Login button clicked.


MANUAL CAPTCHA / SECURITY VERIFICATION
If Instagram displays a CAPTCHA, security check, or verification,
complete it manually in the Chrome window.
You have 30 seconds.
The scraper will NOT attempt to bypass it.
Time remaining: 01 seconds

CAPTCHA/verification window finished.
Checking Instagram login session...
Current URL: https://www.instagram.com/accounts/onetap/


LOGIN CONFIRMED
Authenticated Instagram session detected.


Opening personalized Instagram Explore...
Personalized Explore page loaded.


Single Chrome window established.
No additional browser windows or tabs will be created.


TIME-BASED SCRAPING STARTED
Duration: 10 minutes
Batch size: 10
Source: Personalized Instagram Explore
Browser mode: SINGLE WINDOW / SINGLE TAB


Time remaining: 10.00 minutes

Found 1/10: https://www.instagram.com/p/DbDhOnMT1iu/